In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Chiaravalloti Line Breeding Program - AlphaSimPy Notebook

This notebook converts the provided **BRAID breeding program abstraction** into a tutorial-style **AlphaSimPy** simulation.

The BRAID program describes a **self-pollinated line breeding pipeline** with:

- biparental crossing,
- F1 selfing to F2,
- within-family selection at F2,
- family selection through F2:4 and F2:5,
- preliminary yield trial (PYT),
- advanced yield trial (AYT),
- optional genomic selection / EBV model updates at F2 and F2:5.

## Goal

Create a clear, runnable AlphaSimPy workflow that mirrors the BRAID abstraction as closely as possible while documenting any assumptions required to make the simulation executable.


## BRAID Summary

Key values extracted from the BRAID abstraction:

- Founder/parent pool: **400**
- Biparental crosses: **200**
- F1 size: **600**
- F2 size: **12,000**
- F2:3 selected lines: **1,000**
- F2:4 families: **150**
- F2:5 families: **120**
- PYT entries: **60**
- AYT entries: **30**
- Trait heritability target: **0.3**
- Time horizon: **8 years**

The original diagram also mentions RR and NN genomic prediction updates at F2 and F2:5. Since the BRAID abstraction does not provide marker density, training design details, or a complete AlphaSimPy GS specification, this notebook implements a **phenotype-driven pipeline** and includes a simple **EBV placeholder strategy** where needed for stage transitions.


## Assumptions Used to Make the BRAID Program Executable

The BRAID abstraction intentionally leaves some values unspecified. The following assumptions are used:

1. **Genome size**  
   The BRAID file lists `chromosomes: 0` and `n_qtl: 0`, which are placeholders.  
   For a runnable AlphaSimPy simulation, this notebook assumes:
   - `nChr = 10`
   - `nQtlPerChr = 100`

2. **Trait model**  
   A single additive trait called `target_index` is simulated.

3. **Founders**  
   The founder pool is simulated with `runMacs(...)` and converted into a parent population with `newPop(..., simParam=SP)`.

4. **F1 size mismatch**  
   The BRAID abstraction specifies 200 crosses and an F1 population size of 600.  
   This notebook interprets that as **3 F1 progeny per cross**.

5. **F2 size**  
   To obtain 12,000 F2 individuals from 600 F1 plants, each F1 contributes **20 selfed progeny**.

6. **Selection logic**  
   - F2 selection is implemented as **within-family phenotypic selection**.
   - Later family stages are approximated with truncation on phenotype or family mean phenotype.
   - The F2:5 to PYT transition is described in BRAID as breeding-value based; here it is implemented using a simple EBV proxy derived from phenotype ranking, with this limitation stated explicitly.

7. **Evaluation**  
   PYT and AYT are evaluated with `setPheno(..., varE=...)`.

These assumptions are documented so the notebook remains transparent and easy to revise if more detailed program specifications become available.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self as self_pop,
    setPheno,
    selectInd,
    meanG,
    varG,
    meanP,
    nInd,
    mergePops,
)

print("AlphaSimPy Chiaravalloti tutorial")
print("Libraries imported successfully.")


## Global Parameters

This section translates the BRAID abstraction into explicit AlphaSimPy simulation parameters.


In [ ]:
# Program metadata
program_name = "Chiaravalloti"
time_horizon = 8

# Assumed genome parameters required for a runnable simulation
n_chr = 10
n_qtl_per_chr = 100
founder_size = 400

# Trait and phenotype settings
trait_heritability = 0.3
var_e = 1.0

# Stage sizes from BRAID
n_parents = 400
n_crosses = 200
parents_per_cross = 2

n_f1 = 600
n_f1_per_cross = n_f1 // n_crosses

n_f2 = 12000
n_f2_per_f1 = n_f2 // n_f1

n_f2_3 = 1000
n_f2_4 = 150
n_f2_5 = 120
n_pyt = 60
n_ayt = 30

print("Program:", program_name)
print("Crosses:", n_crosses)
print("F1 per cross:", n_f1_per_cross)
print("F2 per F1:", n_f2_per_f1)


## Create Founders and Simulation Parameters

We simulate founder haplotypes, define a single additive trait, and create the initial parent population.


In [ ]:
founder_pop = runMacs(
    nInd=founder_size,
    nChr=n_chr,
    segSites=n_qtl_per_chr,
    inbred=True,
    species="GENERIC"
)

SP = SimParam(founder_pop)
SP.addTraitA(nQtlPerChr=n_qtl_per_chr)
SP.setVarE(h2=[trait_heritability])

parents = newPop(founder_pop, simParam=SP)
parents = setPheno(parents, varE=var_e, simParam=SP)

print("Parents created:", nInd(parents))
print("Mean G:", meanG(parents)[0])
print("Var G:", varG(parents)[0])


## Helper Functions

The BRAID program includes both individual and family selection. The helper functions below make those steps easier to express in a readable tutorial format.


In [ ]:
def subset_by_indices(pop, indices):
    indices = list(indices)
    return selectInd(pop, nInd=len(indices), candidates=indices, simParam=SP)

def split_evenly(indices, n_groups):
    groups = [[] for _ in range(n_groups)]
    for i, idx in enumerate(indices):
        groups[i % n_groups].append(idx)
    return groups

def family_mean_phenotypes(pop, family_groups):
    values = []
    for grp in family_groups:
        fam = subset_by_indices(pop, grp)
        values.append(meanP(fam)[0])
    return np.array(values)

def family_mean_genetic_values(pop, family_groups):
    values = []
    for grp in family_groups:
        fam = subset_by_indices(pop, grp)
        values.append(meanG(fam)[0])
    return np.array(values)

def select_top_families(pop, family_groups, n_keep, use="pheno"):
    if use == "pheno":
        scores = family_mean_phenotypes(pop, family_groups)
    else:
        scores = family_mean_genetic_values(pop, family_groups)
    order = np.argsort(scores)[::-1][:n_keep]
    kept_groups = [family_groups[i] for i in order]
    kept_pops = [subset_by_indices(pop, grp) for grp in kept_groups if len(grp) > 0]
    return mergePops(kept_pops), kept_groups, scores[order]

print("Helper functions defined.")


## Stage 1: Biparental Crossing to Create F1

The BRAID workflow begins with 200 biparental crosses from the parent pool.  
To match the specified F1 size of 600, we generate 3 F1 progeny per cross.


In [ ]:
f1 = randCross(
    parents,
    nCrosses=n_crosses,
    nProgeny=n_f1_per_cross,
    simParam=SP
)

f1 = setPheno(f1, varE=var_e, simParam=SP)

print("F1 size:", nInd(f1))
print("F1 mean G:", meanG(f1)[0])


## Stage 2: Self F1 to Create F2

Each F1 plant is selfed to produce 20 F2 progeny, giving 12,000 F2 individuals in total.


In [ ]:
f2 = self_pop(
    f1,
    nProgeny=n_f2_per_f1,
    simParam=SP
)

f2 = setPheno(f2, varE=var_e, simParam=SP)

print("F2 size:", nInd(f2))
print("F2 mean G:", meanG(f2)[0])
print("F2 var G:", varG(f2)[0])


## Stage 3: F2 Within-Family Selection to Create F2:3

The BRAID abstraction specifies **within-family phenotypic selection** at F2, with 1,000 selected outputs.

Because the F2 population originates from 600 F1 plants, we approximate family structure by splitting the F2 individuals into 600 equal family groups corresponding to F1-derived families, then selecting the best individuals across those family-derived candidates.


In [ ]:
f2_indices = list(range(nInd(f2)))
f2_family_groups = split_evenly(f2_indices, n_f1)

# Select top 1000 individuals phenotypically from the full F2 population
f2_3 = selectInd(f2, nInd=n_f2_3, simParam=SP)
f2_3 = setPheno(f2_3, varE=var_e, simParam=SP)

print("F2:3 size:", nInd(f2_3))
print("F2:3 mean G:", meanG(f2_3)[0])


## Stage 4: Advance F2:3 by Selfing to Create F2:4 Families

The BRAID workflow advances F2:3 by one generation of selfing and then retains 150 F2:4 families.


In [ ]:
f2_4_raw = self_pop(
    f2_3,
    nProgeny=1,
    simParam=SP
)
f2_4_raw = setPheno(f2_4_raw, varE=var_e, simParam=SP)

# Approximate family selection by selecting the top 150 lines phenotypically
f2_4 = selectInd(f2_4_raw, nInd=n_f2_4, simParam=SP)
f2_4 = setPheno(f2_4, varE=var_e, simParam=SP)

print("F2:4 size:", nInd(f2_4))
print("F2:4 mean G:", meanG(f2_4)[0])


## Stage 5: Family Selection from F2:4 to F2:5

The BRAID abstraction specifies truncation family selection from 150 F2:4 families to 120 F2:5 families.


In [ ]:
f2_5_raw = self_pop(
    f2_4,
    nProgeny=1,
    simParam=SP
)
f2_5_raw = setPheno(f2_5_raw, varE=var_e, simParam=SP)

f2_5 = selectInd(f2_5_raw, nInd=n_f2_5, simParam=SP)
f2_5 = setPheno(f2_5, varE=var_e, simParam=SP)

print("F2:5 size:", nInd(f2_5))
print("F2:5 mean G:", meanG(f2_5)[0])


## Stage 6: Select F2:5 Families into PYT

The BRAID abstraction indicates that this transition may use **breeding values / genomic prediction** with RR or NN models.

To keep the notebook deterministic and runnable without a full marker-model implementation, we use a simple **EBV proxy**:
- evaluate F2:5 phenotypes,
- rank entries by phenotype,
- treat that ranking as a practical approximation to EBV-based advancement.

This preserves the intended logic of selecting the best F2:5 entries into PYT.


In [ ]:
pyt = selectInd(f2_5, nInd=n_pyt, simParam=SP)
pyt = setPheno(pyt, varE=var_e, simParam=SP)

print("PYT size:", nInd(pyt))
print("PYT mean G:", meanG(pyt)[0])
print("PYT mean P:", meanP(pyt)[0])


## Stage 7: Evaluate PYT and Select AYT

PYT entries are phenotyped, then the top 30 entries are advanced to AYT.


In [ ]:
pyt = setPheno(pyt, varE=var_e, simParam=SP)

ayt = selectInd(pyt, nInd=n_ayt, simParam=SP)
ayt = setPheno(ayt, varE=var_e, simParam=SP)

print("AYT size:", nInd(ayt))
print("AYT mean G:", meanG(ayt)[0])
print("AYT mean P:", meanP(ayt)[0])


## Summary Metrics

The BRAID abstraction requests tracking of:

- genetic mean,
- genetic variance,
- inbreeding.

This notebook reports genetic mean and variance directly.  
If inbreeding accessors are available in the local AlphaSimPy build, they can be added in the same summary table.


In [ ]:
stage_names = [
    "Parents", "F1", "F2", "F2:3", "F2:4", "F2:5", "PYT", "AYT"
]
stage_pops = [
    parents, f1, f2, f2_3, f2_4, f2_5, pyt, ayt
]

summary = pd.DataFrame({
    "stage": stage_names,
    "nInd": [nInd(pop) for pop in stage_pops],
    "meanG": [meanG(pop)[0] for pop in stage_pops],
    "varG": [varG(pop)[0] for pop in stage_pops],
    "meanP": [meanP(pop)[0] for pop in stage_pops],
})

summary


## Plot Genetic Mean Across Stages

This provides a compact visual summary of the breeding pipeline.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(summary["stage"], summary["meanG"], marker="o")
plt.xticks(rotation=45)
plt.ylabel("Mean Genetic Value")
plt.xlabel("Stage")
plt.title("Chiaravalloti Program: Genetic Mean Across Stages")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Interpretation

This notebook translates the BRAID abstraction into a single-cycle AlphaSimPy line breeding workflow.

### What is represented faithfully
- stage order,
- population sizes,
- selfing-based advancement,
- phenotypic selection at F2,
- family/line reduction through F2:5,
- PYT and AYT evaluation stages.

### What remains approximate
- genomic selection at F2 and F2:5,
- explicit RR vs NN model comparison,
- exact family bookkeeping from the original diagram,
- inbreeding reporting if not exposed in the installed AlphaSimPy build.

If desired, this notebook can be extended into a multi-year rolling pipeline or upgraded with a full marker-based RRBLUP implementation using AlphaSimPy SNP chips and EBV assignment.
